# 🧩 Step 2: Multi-Task Patch Extraction for Deep Learning
### Prepares 256x256 Image Tiles with Multi-Channel Inputs & Multi-Head Targets

Generates patches into `/content/patches` (fast local NVMe disk on Colab):
- **7-Channel Inputs:** Green, Red, NIR, NDVI, MBI, Road Distance, Swath ID.
- **4 Multi-Task Targets:** Continuous footprint coverage [0-1], Binary mask, Boundary edges, and Ignore mask.
- **Spatial Integrity:** Strictly respects the Block Train / Validation / Test split.

In [ ]:
# 1. Setup Environment & Configuration
import os, sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_DIR = "/content/drive/MyDrive/Encroachment"
    if not os.path.isdir(ROOT_DIR):
        ROOT_DIR = "/content/drive/My Drive/Encroachment"
    sys.path.append(ROOT_DIR)
    %pip install -q rasterio geopandas scipy shapely
else:
    ROOT_DIR = os.getcwd()
    sys.path.append(ROOT_DIR)

import numpy as np
from encroach.config import Config
from encroach.scene import Scene
from encroach.labels import load_split
from encroach.data import extract_patches

cfg = Config(ROOT=ROOT_DIR)
scene = Scene(cfg, load_state=True)
split, cnt = load_split(cfg)
print("Loaded Scene and Spatial Split. Output patch dir:", cfg.PATCH_DIR)

In [ ]:
# 2. Extract Multi-Task Patches
# Patches are extracted with 50% stride (128 px overlap) to ensure complete spatial representation
patch_stats = extract_patches(scene, split, cfg.PATCH_DIR)
print("Extraction complete. Patch summary:", patch_stats)

In [ ]:
# 3. Inspect a Random Patch Sample (Verification)
import glob
import matplotlib.pyplot as plt

train_patches = glob.glob(os.path.join(cfg.PATCH_DIR, "p_train_*_bld1.npz"))
if train_patches:
    sample = np.load(train_patches[0])
    img = sample["x"]
    cov = sample["cov"]
    mask = sample["mask"]
    bnd = sample["boundary"]
    
    fig, ax = plt.subplots(1, 4, figsize=(18, 5))
    # Standard False Color (NIR, Red, Green)
    fcc = np.dstack([img[2], img[1], img[0]])
    fcc = np.clip((fcc - np.percentile(fcc, 2)) / (np.percentile(fcc, 98) - np.percentile(fcc, 2) + 1e-6), 0, 1)
    ax[0].imshow(fcc); ax[0].set_title("LISS-4 False Color (NIR-R-G)")
    ax[1].imshow(cov, cmap="viridis"); ax[1].set_title("Target: Continuous Coverage [0-1]")
    ax[2].imshow(mask, cmap="Reds"); ax[2].set_title("Target: Binary Building Mask")
    ax[3].imshow(bnd, cmap="magma"); ax[3].set_title("Target: Footprint Boundary Edge")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No positive building patches found in patch directory.")